In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
.appName("MyApp")\
.master("local[*]")\
.getOrCreate()
sc = spark.sparkContext 

In [5]:
spark

In [7]:
df = spark.read.format("csv")\
        .option("header","true")\
        .option("inferSchema","true")\
        .load("data/retail-data/by-day/2010-12-01.csv")

df.printSchema()
df.createOrReplaceTempView("dfTable")

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [10]:
from pyspark.sql.functions import lit
df.select(lit(5), lit("five"), lit(5.0))

DataFrame[5: int, five: string, 5.0: double]

In [11]:
df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows



In [14]:
from pyspark.sql.functions import col
df.where(col("InvoiceNo") != 536365)\
        .select("InvoiceNo","Description")\
        .show(5,False)

+---------+-----------------------------+
|InvoiceNo|Description                  |
+---------+-----------------------------+
|536366   |HAND WARMER UNION JACK       |
|536366   |HAND WARMER RED POLKA DOT    |
|536367   |ASSORTED COLOUR BIRD ORNAMENT|
|536367   |POPPY'S PLAYHOUSE BEDROOM    |
|536367   |POPPY'S PLAYHOUSE KITCHEN    |
+---------+-----------------------------+
only showing top 5 rows



In [19]:
df.where("InvoiceNo = 536365")\
    .show(5, False)

df.where("InvoiceNo <> 536365")\
    .show(5, False)


+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850.0   |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
+---------+-----

In [24]:
from pyspark.sql.functions import instr
priceFilter = col("UnitPrice") > 600
descripFilter = instr(df.Description, "POSTAGE") >= 1
df.where(df.StockCode.isin("DOT")).where(priceFilter | descripFilter).show()

+---------+---------+--------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|   Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------+--------+-------------------+---------+----------+--------------+
|   536544|      DOT|DOTCOM POSTAGE|       1|2010-12-01 14:32:00|   569.77|      NULL|United Kingdom|
|   536592|      DOT|DOTCOM POSTAGE|       1|2010-12-01 17:06:00|   607.49|      NULL|United Kingdom|
+---------+---------+--------------+--------+-------------------+---------+----------+--------------+



In [36]:
from pyspark.sql.functions import expr, pow
fabricatedQuantity = pow(col("Quantity")*col("UnitPrice"),2)+5
df.select(expr("customerId"),
         fabricatedQuantity.alias("realquantity")).show(5)


df.selectExpr("CustomerId", "(POWER((Quantity*UnitPrice), 2.0)+5) as realQuantity").show(5)

+----------+------------------+
|customerId|      realquantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
|   17850.0|             489.0|
|   17850.0|          418.7156|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 5 rows

+----------+------------------+
|CustomerId|      realQuantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
|   17850.0|             489.0|
|   17850.0|          418.7156|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 5 rows



In [38]:
from pyspark.sql.functions import lit, round, bround
df.select(round(lit("2.5")), bround(lit("2.5"))).show(3)

+-------------+--------------+
|round(2.5, 0)|bround(2.5, 0)|
+-------------+--------------+
|          3.0|           2.0|
|          3.0|           2.0|
|          3.0|           2.0|
+-------------+--------------+
only showing top 3 rows



In [39]:
from pyspark.sql.functions import initcap
df.select(initcap(col("Description"))).show()

+--------------------+
|initcap(Description)|
+--------------------+
|White Hanging Hea...|
| White Metal Lantern|
|Cream Cupid Heart...|
|Knitted Union Fla...|
|Red Woolly Hottie...|
|Set 7 Babushka Ne...|
|Glass Star Froste...|
|Hand Warmer Union...|
|Hand Warmer Red P...|
|Assorted Colour B...|
|Poppy's Playhouse...|
|Poppy's Playhouse...|
|Feltcraft Princes...|
|Ivory Knitted Mug...|
|Box Of 6 Assorted...|
|Box Of Vintage Ji...|
|Box Of Vintage Al...|
|Home Building Blo...|
|Love Building Blo...|
|Recipe Box With M...|
+--------------------+
only showing top 20 rows



In [43]:
#upper case and lower case
from pyspark.sql.functions import lower,upper
df.select(col("description"), lower(col("description")),upper(lower(col("description")))).show(2)

+--------------------+--------------------+-------------------------+
|         description|  lower(description)|upper(lower(description))|
+--------------------+--------------------+-------------------------+
|WHITE HANGING HEA...|white hanging hea...|     WHITE HANGING HEA...|
| WHITE METAL LANTERN| white metal lantern|      WHITE METAL LANTERN|
+--------------------+--------------------+-------------------------+
only showing top 2 rows



In [45]:
from pyspark.sql.functions import trim, ltrim,rtrim,rpad,lpad,lit
df.select(
    ltrim(lit("   HELLO   ")).alias("ltrim"),
    rtrim(lit("   HELLO   ")).alias("rtrim"),
    trim(lit("   HELLO   ")).alias("trim"),
    lpad(lit("HELLO"),3," ").alias("lpad"),
    rpad(lit("HELLO"), 10, " ").alias("rpad")).show(3)

+--------+--------+-----+----+----------+
|   ltrim|   rtrim| trim|lpad|      rpad|
+--------+--------+-----+----+----------+
|HELLO   |   HELLO|HELLO| HEL|HELLO     |
|HELLO   |   HELLO|HELLO| HEL|HELLO     |
|HELLO   |   HELLO|HELLO| HEL|HELLO     |
+--------+--------+-----+----+----------+
only showing top 3 rows



In [46]:
from pyspark.sql.functions import translate
df.select(translate(col("Description"), "LEET", "1337"),col("Description")).show(2)

+----------------------------------+--------------------+
|translate(Description, LEET, 1337)|         Description|
+----------------------------------+--------------------+
|              WHI73 HANGING H3A...|WHITE HANGING HEA...|
|               WHI73 M37A1 1AN73RN| WHITE METAL LANTERN|
+----------------------------------+--------------------+
only showing top 2 rows



In [51]:
from pyspark.sql.functions import current_date, current_timestamp
dataDF = spark.range(10)\
        .withColumn("today", current_date())\
        .withColumn("now", current_timestamp())
dataDF.createOrReplaceTempView("datetable")
dataDF.printSchema()
dataDF.show(5)

root
 |-- id: long (nullable = false)
 |-- today: date (nullable = false)
 |-- now: timestamp (nullable = false)

+---+----------+--------------------+
| id|     today|                 now|
+---+----------+--------------------+
|  0|2026-06-01|2026-06-01 16:10:...|
|  1|2026-06-01|2026-06-01 16:10:...|
|  2|2026-06-01|2026-06-01 16:10:...|
|  3|2026-06-01|2026-06-01 16:10:...|
|  4|2026-06-01|2026-06-01 16:10:...|
+---+----------+--------------------+
only showing top 5 rows



In [54]:
#add or substract 5 days from today
from pyspark.sql.functions import date_add, date_sub
dataDF.select(date_sub(col("today"), 5), date_add(col("today"), 5),(col("today"))).show(5)

+------------------+------------------+----------+
|date_sub(today, 5)|date_add(today, 5)|     today|
+------------------+------------------+----------+
|        2026-05-27|        2026-06-06|2026-06-01|
|        2026-05-27|        2026-06-06|2026-06-01|
|        2026-05-27|        2026-06-06|2026-06-01|
|        2026-05-27|        2026-06-06|2026-06-01|
|        2026-05-27|        2026-06-06|2026-06-01|
+------------------+------------------+----------+
only showing top 5 rows



In [61]:
from pyspark.sql.functions import datediff, months_between, to_date
dataDF.withColumn("week_ago", date_sub(col("today"), 7))\
        .select(datediff(col("week_ago"), col("today"))).show(5)

dataDF.select(
    to_date(lit("2016-01-01")). alias("start"),
    to_date(lit("2017-05-22")). alias("end"))\
.select(months_between(col("start"), col("end"))).show(3)

+-------------------------+
|datediff(week_ago, today)|
+-------------------------+
|                       -7|
|                       -7|
|                       -7|
|                       -7|
|                       -7|
+-------------------------+
only showing top 5 rows

+--------------------------------+
|months_between(start, end, true)|
+--------------------------------+
|                    -16.67741935|
|                    -16.67741935|
|                    -16.67741935|
+--------------------------------+
only showing top 3 rows



In [62]:
from pyspark.sql.functions import to_date, lit
spark.range(5).withColumn("date", lit("2017-01-01"))\
.select(to_date(col("date"))).show(3)

+-------------+
|to_date(date)|
+-------------+
|   2017-01-01|
|   2017-01-01|
|   2017-01-01|
+-------------+
only showing top 3 rows



In [64]:
dataDF.select(to_date(lit("2016-20-12")), to_date(lit("2017-12-11"))).show(3)

+-------------------+-------------------+
|to_date(2016-20-12)|to_date(2017-12-11)|
+-------------------+-------------------+
|               NULL|         2017-12-11|
|               NULL|         2017-12-11|
|               NULL|         2017-12-11|
+-------------------+-------------------+
only showing top 3 rows



In [70]:
from pyspark.sql.functions import to_date
dateformat = "yyyy-dd-MM"
cleanDateDF = spark.range(1).select(
    to_date(lit("2017-12-11"), dateformat).alias("date"),
    to_date(lit("2017-20-12"), dateformat).alias("date2"))

cleanDateDF.createOrReplaceTempView("dateTable2")
cleanDateDF.show()

+----------+----------+
|      date|     date2|
+----------+----------+
|2017-11-12|2017-12-20|
+----------+----------+



In [71]:
from pyspark.sql.functions import to_timestamp
cleanDateDF.select(to_timestamp(col("date"), dateformat)).show()

+------------------------------+
|to_timestamp(date, yyyy-dd-MM)|
+------------------------------+
|           2017-11-12 00:00:00|
+------------------------------+



In [73]:
# structs are dataframe within dataframe
from pyspark.sql.functions import struct
complexDF = df.select(struct(
    "Description", "InvoiceNo").alias("complex"))
complexDF.createOrReplaceTempView("complexDF")

In [74]:
complexDF.show()

+--------------------+
|             complex|
+--------------------+
|{WHITE HANGING HE...|
|{WHITE METAL LANT...|
|{CREAM CUPID HEAR...|
|{KNITTED UNION FL...|
|{RED WOOLLY HOTTI...|
|{SET 7 BABUSHKA N...|
|{GLASS STAR FROST...|
|{HAND WARMER UNIO...|
|{HAND WARMER RED ...|
|{ASSORTED COLOUR ...|
|{POPPY'S PLAYHOUS...|
|{POPPY'S PLAYHOUS...|
|{FELTCRAFT PRINCE...|
|{IVORY KNITTED MU...|
|{BOX OF 6 ASSORTE...|
|{BOX OF VINTAGE J...|
|{BOX OF VINTAGE A...|
|{HOME BUILDING BL...|
|{LOVE BUILDING BL...|
|{RECIPE BOX WITH ...|
+--------------------+
only showing top 20 rows



In [77]:
complexDF.select("complex.Description")

DataFrame[Description: string]

In [78]:
complexDF.select(col("complex").getField("Description"))

DataFrame[complex.Description: string]

In [79]:
complexDF.select("complex.*")

DataFrame[Description: string, InvoiceNo: string]

In [86]:
from pyspark.sql.functions import split
df.select(split(col("Description"), " ").alias("array_col"))\
    .selectExpr("array_col[0]").show(12)

+------------+
|array_col[0]|
+------------+
|       WHITE|
|       WHITE|
|       CREAM|
|     KNITTED|
|         RED|
|         SET|
|       GLASS|
|        HAND|
|        HAND|
|    ASSORTED|
|     POPPY'S|
|     POPPY'S|
+------------+
only showing top 12 rows



In [85]:
# array lenght
from pyspark.sql.functions import size
df.select(size(split(col("Description"), " "))).show(2)

#if the value exits in the array
from pyspark.sql.functions import array_contains
df.select(array_contains(split(col("Description")," "),"WHITE")).show(2)

+-------------------------------+
|size(split(Description,  , -1))|
+-------------------------------+
|                              5|
|                              3|
+-------------------------------+
only showing top 2 rows

+------------------------------------------------+
|array_contains(split(Description,  , -1), WHITE)|
+------------------------------------------------+
|                                            true|
|                                            true|
+------------------------------------------------+
only showing top 2 rows



In [89]:
from pyspark.sql.functions import explode
df.withColumn("splitted", split(col("Description"), " "))\
	.withColumn("exploded", explode(col("splitted")))\
    .select("Description", "InvoiceNo", "exploded").show(5)

+--------------------+---------+--------+
|         Description|InvoiceNo|exploded|
+--------------------+---------+--------+
|WHITE HANGING HEA...|   536365|   WHITE|
|WHITE HANGING HEA...|   536365| HANGING|
|WHITE HANGING HEA...|   536365|   HEART|
|WHITE HANGING HEA...|   536365| T-LIGHT|
|WHITE HANGING HEA...|   536365|  HOLDER|
+--------------------+---------+--------+
only showing top 5 rows



In [94]:
# working with json
jsonDF = spark.range(1).selectExpr("""'{"myJSONKey" : {"myJSONValue" : [1, 2, 3]}}' as jsonString""")

In [98]:
from pyspark.sql.functions import col, get_json_object, json_tuple

jsonDF.select(
    get_json_object(
        col("jsonString"),
        "$.myJSONKey.myJSONValue[1]"
    ).alias("column"),
    json_tuple(col("jsonString"), "myJSONKey")
).show(2)

+------+--------------------+
|column|                  c0|
+------+--------------------+
|     2|{"myJSONValue":[1...|
+------+--------------------+

